# Collecting Human Preferences with the Prolific AI Task Builder

This notebook demonstrates how to use the **Prolific AI Task Builder** to transform LLM-generated responses into structured human preference data — a crucial step in **training and evaluating reward models**.

The **Prolific AI Task Builder** provides a streamlined way to:
- Present multiple model responses to participants
- Collect reliable human preferences at scale
- Export results in a standardized format ready for machine learning workflows

By the end of this notebook, we will:
1. Load a set of LLM-generated prompts and responses  
2. Use the **AI Task Builder** to collect preference data from Prolific participants  
3. Convert the collected data into tuples of the form `("prompt", "chosen", "rejected")` the same structure used in TRL (Hugging Face), OpenAI PPO, and Anthropic's Constitutional AI pipelines.

This workflow illustrates how **Prolific’s infrastructure** can directly support **RLHF (Reinforcement Learning from Human Feedback)** and **AI alignment research**, bridging human feedback collection and model training in a reproducible, scalable way.

In [1]:
import os, json, time, yaml, requests
import secrets, string
from pathlib import Path
import pandas as pd
from io import StringIO

In [2]:
# Read API tokens for Jupyter Notebook
from dotenv import load_dotenv
load_dotenv()

True

In [348]:
# Set paths
data_dir = Path('.')
output_dir = data_dir / 'outputs'

# Read
pairs_csv = output_dir / 'response_pairs.csv'
config_path = Path('..') / 'examples' / 'config.yaml'

# Write
raw_demographic_csv = output_dir / 'raw_demographic.csv'
demographic_csv = output_dir / 'demographic.csv'
raw_preferences_csv = output_dir / 'raw_preferences.csv'
votes_preferences_csv = output_dir / 'votes_preferences.csv'
preferences_jsonl = output_dir / 'preferences.jsonl'

In [285]:
# Load config
with open(config_path, 'r') as f:
    cfg = yaml.safe_load(f)

In [5]:
# Load completions
df = pd.read_csv(pairs_csv)
df.head()

,Prompt,Response A,Response B
0,How do I make homemade pasta from scratch?,"To make fresh pasta dough, combine flour, eggs...",Making your own pasta dough is easier than you...
1,How do I make homemade pasta from scratch?,"To make fresh pasta dough, combine flour, eggs...",How to make homemade pasta from scratch 1. Mix...
2,How do I make homemade pasta from scratch?,"To make fresh pasta dough, combine flour, eggs...",How to Make Fresh Pasta Dough. Add 2 to 3 tabl...
3,How do I make homemade pasta from scratch?,Making your own pasta dough is easier than you...,How to make homemade pasta from scratch 1. Mix...
4,How do I make homemade pasta from scratch?,Making your own pasta dough is easier than you...,How to Make Fresh Pasta Dough. Add 2 to 3 tabl...


In [6]:
num_tasks = df.shape[0]
print(f"There's {df.shape[0]} tasks.")

(24, 3)

## Authenticate on Prolific

In [7]:
prolific_token = os.environ.get("PROLIFIC_API_TOKEN")

base = "https://api.prolific.com/api/v1"

headers = {
    "Authorization": f"Token {prolific_token}",
    "Content-Type": "application/json",
}

In [8]:
# Sanity check: fetch your Prolific researcher ID
res = requests.get(f"{base}/users/me/", headers=headers)
researcher_id = res.json()["id"]
researcher_name = res.json()['name']

assert isinstance(researcher_id, str) and len(researcher_id) == 24 and all(c in '0123456789abcdef' for c in researcher_id), \
    f"Invalid or missing researcher_id: {researcher_id}"

In [9]:
workspace_id = os.environ.get("PROLIFIC_WORKSPACE_ID")

# AI Task Builder
AI Task Builder is a tool designed to streamline the process of creating data annotation tasks that integrate seamlessly with the Prolific participant pool.

Key Features:
- Multiple choice and free text annotation inputs
- Seamless integration with Prolific Taskflow and the Prolific participant pool

Example of the AI Task Builder API workflow:

1. **Create a dataset**: Initialize a dataset record for your project.
2. **Upload your data**: Request presigned URLs for your files, then upload the data directly to S3 using the presigned URLs.
3. **Monitor dataset status**: Wait for the dataset to be ready before proceeding.
4. **Define task details**: Specify the schema that describes what annotators will see and choose.
5. **Set up the batch**: Configure the batch with necessary details.
6. **Create instructions**: Define the guidelines for your annotation tasks.
7. **Monitor batch status**: Ensure the batch is ready attaching to a study.
8. **Create a study**: Create a study that references your configured AI Task Builder batch. See the example request to further illustrate this step.
9. **Publish the study**: Publish the study to distribute tasks to annotators.
10. **Wait for task completion**: Allow time for annotators to complete the assigned tasks.
11. **Fetch responses**: Retrieve the completed annotations for your batch.

### 1. Create a dataset: Initialize a dataset record for your project.

In [10]:
payload = {
    "name": cfg['prolific']['batch_name'],
    "workspace_id": workspace_id
}

resp = requests.post(
    f"{base}/data-collection/datasets", 
    headers=headers,
    data=json.dumps(payload)
)

resp.raise_for_status()  # will raise on 4XX
data = resp.json()

In [12]:
dataset_id = data['id']

### 2. Upload your data: Request presigned URLs for your files, then upload the data directly to S3 using the presigned URLs.

In [13]:
# Get presigned URL
resp = requests.get(
    f"{base}/data-collection/datasets/{dataset_id}/upload-url/{pairs_csv.name}",
    headers=headers
)

resp.raise_for_status()  # will raise on 4XX
upload_info = resp.json()

In [14]:
upload_url = upload_info["upload_url"]

In [15]:
# Upload file to S3 using PUT
with open(pairs_csv, "rb") as f:
    put_resp = requests.put(upload_url, data=f)

put_resp.raise_for_status()  # will raise on 4XX

### 3. Monitor dataset status: Wait for the dataset to be ready before proceeding.

In [16]:
# Monitor dataset status until READY
while True:
    status_resp = requests.get(
        f"{base}/data-collection/datasets/{dataset_id}/status", 
        headers=headers
    )
    status_resp.raise_for_status()  # will raise on 4XX
    status = status_resp.json().get("status")
    print("Dataset status:", status)
    if status in ("READY", "ERROR"):
        break
    time.sleep(5)

if status == "READY":
    print("✅ Dataset processing complete and ready to use.")
else:
    print("❌ Dataset processing failed. Check your file format or contact Prolific support.")

Dataset status: PROCESSING
Dataset status: READY
✅ Dataset processing complete and ready to use.


### 4. Define task details: Specify the schema that describes what annotators will see and choose.

In [45]:
# Task schema for pairwise A/B preference
task_details = {
    # human-facing fields
    "task_name": cfg['prolific']['task_schema']['task_name'],
    "task_introduction": cfg['prolific']['task_schema']['task_introduction'].replace("\n", " "),
    "task_steps": cfg['prolific']['task_schema']['task_steps'].replace("\n", " "),
    
    # machine-facing fields
    "inputs": [
        {"key": "prompt",     "label": "Prompt"},
        {"key": "response_a", "label": "Response A"},
        {"key": "response_b", "label": "Response B"},
    ],
    "judgment": {
        "type": "single_choice",
        "options": [
            {"value": "A", "label": "Choose Response A"},
            {"value": "B", "label": "Choose Response B"},
        ]
    },
    "randomize_inputs": ["Response A", "Response B"],  # randomize display order
    "validation": {"require_choice": True}
}

### 5. Set up the batch: Configure the batch with necessary details.

In [46]:
payload = {
    "name": cfg['prolific']['batch_name'],
    "workspace_id": workspace_id,
    "dataset_id": dataset_id,
    "task_details": task_details
}

resp = requests.post(
    f"{base}/data-collection/batches", 
    headers=headers, 
    data=json.dumps(payload)
)

resp.raise_for_status()  # will raise on 4XX
batch = resp.json()

In [48]:
resp.json()['status']

'UNINITIALISED'

In [49]:
batch_id = batch["id"]

### 6. Create instructions: Define the guidelines for your annotation tasks.

In [50]:
instructions_payload = {
    "instructions": [   
        {
            "type": "multiple_choice",
            "created_by": researcher_name,
            "description": cfg['prolific']['task_schema']['task_question'],
            "options": [
                {"label": "Response A is better", "value": "A"},
                {"label": "Response B is better", "value": "B"},
            ]
        }
    ]
}

In [51]:
resp = requests.post(
    f"{base}/data-collection/batches/{batch_id}/instructions",
    headers=headers,
    data=json.dumps(instructions_payload)
)

resp.raise_for_status() # will raise on 4XX

### 7. Monitor batch status: Ensure the batch is ready attaching to a study.

In [39]:
# After this step, you can't edit the batch
# 1) Kick off setup (async) — batch will move UNINITIALISED -> PROCESSING
resp = requests.post(
    f"{base}/data-collection/batches/{batch_id}/setup",
    headers=headers,
    data=json.dumps(
        {"dataset_id": dataset_id, 
         "tasks_per_group": cfg['prolific']['task_schema']['tasks_per_group'],
        }
    )
)

resp.raise_for_status() # will raise on 4XX

In [40]:
# 2) Poll until READY (or ERROR)
timeout_sec = 900
poll_sec = 5
t0 = time.time()
last = None

while True:
    resp = requests.get(
        f"{base}/data-collection/batches/{batch_id}/status", 
        headers=headers
    )
    resp.raise_for_status()  # will raise on 4XX
    
    status = resp.json().get("status")
    if status != last:
        print("Batch status:", status)
        last = status
    if status in ("READY", "ERROR"):
        break
    if time.time() - t0 > timeout_sec:
        raise TimeoutError("Batch setup did not finish in time.")
    time.sleep(poll_sec)

if status == "READY":
    # (optional) fetch batch to see total_task_count
    resp = requests.get(f"{base}/data-collection/batches/{batch_id}", headers=headers)
    resp.raise_for_status()  # will raise on 4XX
    print("✅ Batch READY. total_task_count:", resp.json().get("total_task_count"))
else:
    print("❌ Batch setup failed. Check dataset and task_details.")

Batch status: READY
✅ Batch READY. total_task_count: 24


### 8. Create a study: Create a study that references your configured AI Task Builder batch. See the example request to further illustrate this step.

In [111]:
# generate a short random code like 'K7Q3X9W2'
_alnum = string.ascii_uppercase + string.digits
completion_code = "".join(secrets.choice(_alnum) for _ in range(8))

payload = {
    "name": cfg["prolific"]["task_schema"]["task_name"],
    "internal_name": cfg["prolific"]["study_setup"]["internal_name"],
    "description": cfg["prolific"]["task_schema"]["task_introduction"].replace("\n", " "),
    
    # AI Task Builder
    "data_collection_method": cfg["prolific"]["study_setup"]["data_collection_method"],
    "data_collection_id": batch_id,
    
    # Study settings
    "project": os.environ.get("PROLIFIC_PROJECT_ID"),
    "estimated_completion_time": cfg["prolific"]["study_setup"]["estimated_completion_time"],
    "max_time": cfg["prolific"]["study_setup"]["max_time"],
    "total_available_places": 1, # placeholder, we have to patch it later
    "reward": cfg["prolific"]["study_setup"]["reward"],
    "device_compatibility": cfg["prolific"]["study_setup"]["device_compatibility"],
    
    # Completion code with auto-approve
    "completion_option": "code",
    "completion_codes": [
        {
            "code": completion_code,
            "code_type": "COMPLETED",
            "actions": [{"action": "AUTOMATICALLY_APPROVE"}],
        }
    ],
    
    # Filter to AI Taskers group: Comparative reasoning
    "filters": [
        {
            "filter_id": "comparative-reasoning",
            "selected_values": ["0"]
        }
    ],
}

In [112]:
# Create draft study
resp = requests.post(
    f"{base}/studies/", 
    headers=headers, 
    data=json.dumps(payload)
)

resp.raise_for_status() # will raise on 4XX
study = resp.json()

In [115]:
study_id = study['id']

In [130]:
# The number of participants per task has to be patched into the study
access_details = [
    {
        "external_url": ad["external_url"],
        # keep allocated as-is, just change total_allocation
        "total_allocation": cfg["prolific"]["study_setup"]["participants_per_task"],
        "allocated": ad["allocated"],
    }
    for ad in study.get("access_details", [])
]

In [158]:
# PATCH the study. Many fields are required on update; reuse from previous POST.
payload = {
    "internal_name": study["internal_name"],
    "name": study["name"],
    "description": study["description"],

    "reward": int(study["reward"]), 
    "total_available_places": len(access_details) * cfg["prolific"]["study_setup"]["participants_per_task"], # main patch

    "study_labels": study["study_labels"],

    "data_collection_metadata": {
        "annotators_per_task": cfg["prolific"]["study_setup"]["participants_per_task"],
        "total_task_groups": len(access_details),
    },

    "audience": study.get("audience", "standard_sample"),
    "device_compatibility": study["device_compatibility"],
    "peripheral_requirements": study.get("peripheral_requirements", []),

    "estimated_completion_time": study["estimated_completion_time"],
    "maximum_allowed_time": study["maximum_allowed_time"],

    "content_warnings": study.get("content_warnings", []),
    "pii": study.get("pii", {"enabled": False}),
    "is_custom_screening": study.get("is_custom_screening", False),

    "filters": study["filters"],

    "study_type": study["study_type"],
    "completion_codes": study["completion_codes"],

    "data_collection_method": study["data_collection_method"],
    "data_collection_id": study["data_collection_id"],

    "access_details": access_details,
    "access_details_collection_id": study["access_details_collection_id"],

    "submissions_config": study["submissions_config"],
}


In [ ]:
resp = requests.patch(
    f"{base}/studies/{study_id}/", 
    headers=headers, 
    data=json.dumps(payload)
)

resp.raise_for_status()  # will raise on 4XX

### 9. Publish the study: Publish the study to distribute tasks to annotators.

In [162]:
resp = requests.post(
    f"{base}/studies/{study_id}/transition/", 
    headers=headers,
    data=json.dumps({"action": "PUBLISH"})
)

resp.raise_for_status()  # will raise on 4XX

### 10. Wait for task completion: Allow time for annotators to complete the assigned tasks.

In [173]:
timeout_sec = 3600 * 6  # 6 hours max wait (adjust as needed)
poll_sec = 60           # check every 60 seconds (adjust as needed)

t0 = time.time()
last = None

while True:
    resp = requests.get(f"{base}/studies/{study_id}/", headers=headers)
    resp.raise_for_status()
    
    study = resp.json()
    status = study.get("status")
    places_taken = study.get("places_taken")
    total_places = study.get("total_available_places")

    print(f"Study status: {status} ({places_taken}/{total_places} places filled)")
 
    if status in ("COMPLETED", "FINISHED", "AWAITING_REVIEW", "CLOSED"):
        print("✅ Study completed!")
        break

    if time.time() - t0 > timeout_sec:
        raise TimeoutError("⏰ Study did not complete within the expected timeframe.")
    
    time.sleep(poll_sec)

Study status: COMPLETED (20/20 places filled)
✅ Study completed!


### 11. Fetch responses: Retrieve the completed annotations for your batch.

In [255]:
report_url = requests.get(
    f"{base}/data-collection/batches/{batch_id}/report",
    headers=headers)

report_url.raise_for_status()

resp = requests.get(report_url.json()["url"])
resp.raise_for_status()  # will raise on 4XX

In [323]:
df_report = pd.read_csv(StringIO(resp.text))
df_report.to_csv(raw_preferences_csv, index=False)

In [324]:
# Keep relevant columns
annotator_pattern = r'^Annotator\d+_(ID|Response)$'
extra_cols = ['Prompt', 'Response A', 'Response B']

df_report = df_report[df_report.columns[df_report.columns.isin(extra_cols) | df_report.columns.str.match(annotator_pattern)]]

In [325]:
# Make sure we collected all the data
assert df_report.shape[0] == num_tasks # All preference pairs
assert int(df_report.columns[-1].split("_")[0].replace("Annotator", "")) == cfg["prolific"]["study_setup"]["participants_per_task"] # Number of annotators we indicated per task

In [330]:
df_report.head(3)

,Prompt,Response A,Response B,Annotator1_ID,Annotator1_Response,Annotator2_ID,Annotator2_Response,Annotator3_ID,Annotator3_Response,Annotator4_ID,Annotator4_Response,Annotator5_ID,Annotator5_Response
0,How do I make homemade pasta from scratch?,Making your own pasta dough is easier than you...,How to make homemade pasta from scratch 1. Mix...,59dd90f6e75b450001a68dac,B,668a41df847a9640d0ff75f5,B,628e0063fa56e316da2a71cb,B,652c0d5b43610c9446899eb5,A,60d2cf988de167c9da56eeb0,B
1,Can you explain how photosynthesis works in si...,I want to teach it to my 9 year old son.\nA pl...,"What is a cell wall, and what does it do? How ...",63bd81465a7246c0da98e4bd,B,65a708e73e47380843931ad8,A,667335f0570884e306616b2a,A,66114a0b05f2eab8dffe8205,A,667af145fa8225b6be0e79b8,A
2,What are some effective strategies for managin...,This article discusses several practical and p...,"There are many, but a few that stand out from ...",66c0f6b657bd4b8a29485652,A,62a889993e01c9e01ee7066e,A,5db303e4863031000b5f0311,A,607cbf385d3202721e3f719f,A,559c3e07fdf99b32b55f2d8d,A


In [333]:
# Count votes
annotator_response_cols = [col for col in df_report.columns if "Annotator" in col and "Response" in col]

def count_votes(row):
    votes = row[annotator_response_cols].dropna().tolist()
    a_wins = sum(v == "A" for v in votes)
    b_wins = sum(v == "B" for v in votes)
    return pd.Series({"A_wins": a_wins, "B_wins": b_wins})

vote_counts = df_report.apply(count_votes, axis=1)
df_votes = pd.concat([df_report[["Prompt", "Response A", "Response B"]], vote_counts], axis=1)

In [337]:
df_votes.head()

,Prompt,Response A,Response B,A_wins,B_wins
0,How do I make homemade pasta from scratch?,Making your own pasta dough is easier than you...,How to make homemade pasta from scratch 1. Mix...,1,4
1,Can you explain how photosynthesis works in si...,I want to teach it to my 9 year old son.\nA pl...,"What is a cell wall, and what does it do? How ...",4,1
2,What are some effective strategies for managin...,This article discusses several practical and p...,"There are many, but a few that stand out from ...",5,0
3,How do I make homemade pasta from scratch?,"To make fresh pasta dough, combine flour, eggs...",How to Make Fresh Pasta Dough. Add 2 to 3 tabl...,4,1
4,What are the key differences between machine l...,There are a few key differences between machin...,What is their significance in our lives? Which...,3,2


In [338]:
df_votes.to_csv(votes_preferences_csv, index=False)

In [ ]:
# Build final JSONL with majority vote

In [339]:
def majority_label(row):
    if row["A_wins"] > row["B_wins"]:
        return row["Response A"], row["Response B"]
    elif row["B_wins"] > row["A_wins"]:
        return row["Response B"], row["Response A"]
    else:
        return None, None

final_rows = []
for _, row in df_votes.iterrows():
    chosen, rejected = majority_label(row)
    if chosen and rejected:  # skip ties
        final_rows.append({
            "prompt": row["Prompt"],
            "chosen_response": chosen,
            "rejected_response": rejected
        })

In [342]:
# Save as JSONL
with open(preferences_jsonl, "w", encoding="utf-8") as f:
    for item in final_rows:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

In [345]:
# Get demographics
resp = requests.get(
    f"{base}/studies/{study_id}/export/",
    headers=headers
)

resp.raise_for_status() # will raise on 4XX

df_demo = pd.read_csv(StringIO(resp.text))

In [349]:
df_demo = df_demo[~df_demo['Completed at'].isna()] # Keep participants that completed the tasks
df_demo.to_csv(raw_demographic_csv, index=False) 

In [355]:
cols_to_keep = ['Participant id', 'Completed at', 'Time taken',
    'Age', 'Sex', 'Language',
    'Country of birth', 'Country of residence', 'Nationality',
    'Ethnicity simplified', 'Fluent languages', 
    'Highest education level completed', 'Student status', 'Degree subject', 
    'Employment status', 'Work role', 
    'Long-term health condition/disability',  'Sexual orientation', 
   'Submission approval rate', 'Total approvals']

df_demo[cols_to_keep].to_csv(demographic_csv, index=False) 

In [356]:
df_demo[cols_to_keep].head(2)

,Participant id,Completed at,Time taken,Age,Sex,Language,Country of birth,Country of residence,Nationality,Ethnicity simplified,Fluent languages,Highest education level completed,Student status,Degree subject,Employment status,Work role,Long-term health condition/disability,Sexual orientation,Submission approval rate,Total approvals
0,66c0f6b657bd4b8a29485652,2025-10-24T23:16:28.164000Z,742.0,36,Female,French,Tunisia,France,Tunisia,Mixed,"Arabic, French, English",Doctorate degree (PhD/other),No,Other,Full-Time,Manager,No,bisexual,100,1081
1,5dd2d431181abc2ecde6edb4,2025-10-24T23:15:14.396000Z,631.0,69,Female,English,Canada,Canada,Canada,White,English,Undergraduate degree (BA/BSc/other),No,Journalism & Information Business,"Not in paid work (e.g. homemaker', 'retired or...",Individual contributor / Non-manager,No,heterosexual,100,2179
